# Ejercicio PCA (Principal Component Analysis)
En este ejercicio vas a trabajar con un dataset de información de ciudadanos como el estado civil, número de hijos, qué gastos e ingresos tiene, etc...

Se cuenta con un target, que es si el ciudadano va a alquilar o a comprar una vivienda. Para ello, con PCA

### Importamos librerias
Principales librerías que usarás durante el notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

### Cargamos datos de entrada
1. Carga el csv *comprar_alquilar.csv*
2. Obtén la información básica: columnas, dimensiones, descripción de las variables, missings...

In [ ]:
df = pd.read_csv('./data/comprar_alquilar.csv')

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
print(df.isnull().sum().sum())

In [ ]:
df.value_counts('estado_civil')

In [ ]:
df.value_counts('trabajo')

In [ ]:
df.groupby('trabajo')['ingresos'].mean()

In [ ]:
df.corr('pearson')

In [ ]:
df['trabajo'].corr(df['ingresos'], method='pearson')

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlación de Pearson')
plt.show()

### Visualicemos las dimensiones
Realiza un análisis univariante. Realiza la gráfica que consideres para cada variable, diferenciando por colores el target (*comprar*).

In [ ]:
num_cols = df.select_dtypes(include='number').columns.drop('comprar', errors='ignore')

n_cols = 2
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.ravel()

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='comprar', kde=True, ax=axes[i], bins=30)
    axes[i].set_title(f'Distribución de {col}')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, axis = plt.subplots(5,2, figsize=(15,20))

alquiler = df[df['comprar']==0] 
comprar = df[df['comprar']==1] 

axes = axis.ravel()
for i in range(len(df.columns)):
    axes[i].hist(alquiler.values[:,i], bins=40, color = 'r', alpha=0.5)
    axes[i].hist(comprar.values[:,i], bins=40, color = 'g', alpha=0.5)
    axes[i].set_title(df.columns[i])

axes[0].legend(["alquiler", "comprar"])

In [ ]:
corr_matrix = df.corr(method='pearson')

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, 
            annot=True,           
            cmap='coolwarm',      
            fmt='.2f',            
            center=0,             
            square=True,          
            linewidths=0.5,       
            cbar_kws={'label': 'Coeficiente de Pearson'})

plt.title('Matriz de Correlación de Pearson', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

## Estandariza los datos
Como el objetivo de estos primeros apartados no es encontrar el mejor modelo con el mejor accuracy, por sencillez, no es necesario dividir en train y test.

In [ ]:
X = df.drop(columns=['comprar'])
y = df['comprar']

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
print(X_scaled_df.head())
print(f"\n{X_scaled_df.mean().values[:3]}")
print(f"{X_scaled_df.std().values[:3]}")

## Aplicamos PCA
Aplica el algoritmo de PCA para 9 components, es decir, para todas las features

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=9)
X_pca = pca.fit_transform(X_scaled)

print("Shape después de PCA:", X_pca.shape)
print("Varianza explicada por cada PC:\n", pca.explained_variance_ratio_)

### Veamos cómo de buenos predictores son las nuevas dimensiones
Responde a las siguientes preguntas:
1. ¿Cuánta varianza explica cada Principal Component?
2. ¿Y de manera acumulada empezando por el PC1?
3. ¿Cuánta varianza explicarían sólo 5 componentes?

In [ ]:
explained_variance = pca.explained_variance_ratio_

In [ ]:
print("Varianza explicada por cada PC:")
for i, var in enumerate(explained_variance, 1):
    print(f"PC{i}: {var*100:.2f}%")

In [ ]:
cumulative_variance = np.cumsum(explained_variance)

print("Varianza explicada acumulada:")
for i, cum_var in enumerate(cumulative_variance, 1):
    print(f"PC1-PC{i}: {cum_var*100:.2f}%")

In [ ]:
varianza_5_pc = cumulative_variance[4]
print(f"Los primeros 5 componentes explican: {varianza_5_pc*100:.2f}% de la varianza total")

### Graficamos la variacion explicada acumulada
Representa en un diagrama de lineas la evolución de la varianza acumulada en función de los PC

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(explained_variance)+1), explained_variance*100)
plt.xlabel('Componente Principal')
plt.ylabel('Varianza explicada (%)')
plt.title('Varianza por PC')
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance*100, marker='o')
plt.axhline(y=70, color='r', linestyle='--', label='70% umbral')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza acumulada (%)')
plt.title('Varianza acumulada')
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

### Veamos la correlación entre las nuevas componentes y las dimensiones originales
Representa en un mapa de calor los PCA vs las variables originales. Esta información la puedes obtener del atributo de PCA *components_*.

In [ ]:
feature_names = df.drop(columns=['comprar']).columns

plt.figure(figsize=(15, 10))
plt.matshow(pca.components_.T, fignum=1, cmap='coolwarm', aspect='auto')
plt.yticks(range(len(feature_names)), feature_names)
plt.xticks(range(pca.n_components_))
plt.xlabel('Componentes Principales')
plt.ylabel('Variables Originales')
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
df_pca_components = pd.DataFrame(
    pca.components_, 
    columns=X.columns,  
    index=[f'PC{i+1}' for i in range(pca.n_components_)]
)

df_pca_components

## Predicciones
1. Divide en train y test
2. Prepara un pipeline compuesto por:
    - StandardScaler,
    - PCA de 5 componentes
    - Un clasificador
3. Entrena
4. Predice con test
5. Calcula el accuracy score en train y test
6. Representa la matriz de confusión
7. ¿Qué combinación de parámetros y componentes mejoraría el accuracy en test?
8. Vuelve a iterar de nuevo con un gridsearch
9. Guarda tu mejor modelo

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(k='all')),
    ('pca', PCA(n_components=5)),
    ('classifier', RandomForestClassifier(max_depth=5))
])

pipeline

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = pipeline.predict(X_test)

print(f"accuracy_score {accuracy_score(y_test, y_pred)}")
print(f"confusion_matrix\n {confusion_matrix(y_test, y_pred)}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [ ]:
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=5)),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
print(f"LogisticRegression accuracy: {accuracy_score(y_test, y_pred_lr)}")

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'pca__n_components': [3, 5, 7, 9],
    'classifier__C': [0.1, 1, 10],
    'classifier__max_iter': [1000]
}

grid_search = GridSearchCV(pipeline_lr, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)
print(f"Best params: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_}")

In [ ]:
import joblib
joblib.dump(grid_search.best_estimator_, 'final_model.pkl')